<a href="https://colab.research.google.com/github/kp-algo/encoder-decoder/blob/main/encoder_decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tokenizers
!pip install datasets
#packages
import numpy as np
import urllib.request, zipfile
import torch
import torch.nn as nn
import torch.nn.functional as f
from tokenizers import ByteLevelBPETokenizer, pre_tokenizers
from tokenizers.pre_tokenizers import Whitespace
import itertools
from datasets import load_dataset

#Load Dataset

In [ ]:
n_train_stories, n_val_stories = 50_000, 5_000     # ~tens of MB of text, plenty for a 3M-param model

def grab(split, n):
    ds = load_dataset("roneneldan/TinyStories", split=split, streaming=True)
    return [ex["text"] for ex in itertools.islice(ds, n)]

train_stories = grab("train", n_train_stories)
val_stories   = grab("validation", n_val_stories)

print(len(train_stories), len(val_stories))
print(train_stories[0][:300])

50000 5000
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and 


#Tokenisation

In [ ]:
EOS = "<eos>"
train_text = EOS.join(train_stories)
val_text   = EOS.join(val_stories)

In [ ]:
tokenizer = ByteLevelBPETokenizer()

tokenizer.train_from_iterator(
    train_stories, vocab_size=3000, min_frequency=2,
    special_tokens=["<bos>", "<mask>", EOS],
)

encode = lambda s: tokenizer.encode(s).ids
decode = lambda ids: tokenizer.decode(ids)

vocab_size = tokenizer.get_vocab_size()
bos_id, mask_id, eos_id = (tokenizer.token_to_id(t) for t in ["<bos>", "<mask>", EOS])

train_data = torch.tensor(encode(train_text), dtype=torch.long)
val_data   = torch.tensor(encode(val_text),   dtype=torch.long)
print(vocab_size, len(train_data), len(val_data))

3000 11745267 1061769


#Hyperparameters

In [ ]:
#hyperparams
torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_embd = 192
block_size = 128
dropout = 0.2
head_size = 48
n_head = 4
n_layers = 6
vocab_size = tokenizer.get_vocab_size()
batch_size = 32
learning_rate = 3e-4
src_len = 128            # characters the encoder reads
tgt_len = 128
max_iters = 35000
eval_interval = 500
eval_iters = 100

# Data Split



*   BART: requries two different inputs stream(one for encoder and one for decoder)
*   GPT: requires a single input stream(for decoder alone)



In [ ]:
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - src_len - tgt_len - 1, (batch_size,))
    src = torch.stack([d[i : i + src_len] for i in ix])
    tgt = torch.stack([d[i + src_len : i + src_len + tgt_len] for i in ix])
    bos = torch.full((batch_size, 1), bos_id, dtype=torch.long)
    dec_in = torch.cat([bos, tgt[:, :-1]], dim=1)
    return src.to(device), dec_in.to(device), tgt.to(device)

In [ ]:
def get_batch_gpt(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i : i + block_size] for i in ix])
    y = torch.stack([d[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

In [ ]:
def mask_batch(x, p=0.15):
    m = torch.rand(x.shape, device=x.device) < p
    inp = x.clone();  inp[m] = mask_id # hide 15% of characters
    tgt = x.clone();  tgt[~m] = -100  # loss only on the hidden ones
    return inp, tgt

#Architecture


*   Head: single attention head; fundamental unit of the architecture
*   MultiHead: stacks the multiple heads
*   FeedForward: simple linear neural network
*   Encoder Block: encoder; no masking
*   Decoder Block: decoder; masking
*   GPT: encoder-decoder; final model







In [ ]:
class Head(nn.Module):
    def __init__(self, head_size, causal):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
        self.causal = causal

    def forward(self, x, context=None):
        B,T,C = x.shape

        q = self.query(x)

        cxt = x if context is None else context

        k = self.key(cxt)
        v = self.value(cxt)

        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        if self.causal:
            wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        wei = f.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out


In [ ]:
class MultiHead(nn.Module):
    def __init__(self, head_size, num_heads, causal):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, causal) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, context=None):
        out = torch.cat([head(x, context) for head in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)

        return out

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd * 4),
            nn.ReLU(),
            nn.Linear(n_embd * 4, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        out = self.net(x)
        return out

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd//n_head
        self.sa_head = MultiHead(head_size, n_head, causal=False)
        self.fwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa_head(self.ln1(x))
        x = x + self.fwd(self.ln2(x))
        return x

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd//n_head
        self.sa_head = MultiHead(head_size, n_head, causal = True)
        self.ca_head = MultiHead(head_size, n_head, causal = False)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)
        self.fwd = FeedForward(n_embd)

    def forward(self, x, enc):
        x = x + self.sa_head(self.ln1(x))
        x = x + self.ca_head(self.ln2(x), enc)
        x = x + self.fwd(self.ln3(x))
        return x

#BART:


---


*   an autoregressive model with both **encoder** and **decoder** block connected with cross attention.
*   requires two separate inputs; one for encoder and other one for decoder.



In [ ]:
class BART(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embd = nn.Embedding(vocab_size, n_embd)
        self.pos_embd = nn.Embedding(block_size, n_embd)

        self.enc_blocks = nn.Sequential(*[EncoderBlock(n_embd, n_head) for _ in range(n_layers//2)])
        self.dec_blocks = nn.ModuleList([DecoderBlock(n_embd, n_head) for _ in range(n_layers//2)])

        self.ln_enc = nn.LayerNorm(n_embd)
        self.lnf = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def embed(self, ids):
        T = ids.shape[1]
        return self.token_embd(ids) + self.pos_embd(torch.arange(T, device=ids.device))

    def encode(self, src):
        return self.ln_enc(self.enc_blocks(self.embed(src)))

    def decode(self, dec_in, enc):
        x = self.embed(dec_in)
        for dec in self.dec_blocks:
            x = dec(x, enc)
        return self.lm_head(self.lnf(x))

    def forward(self, src, dec_in, targets=None):
        logits = self.decode(dec_in, self.encode(src))
        loss = None if targets is None else f.cross_entropy(
            logits.view(-1, logits.shape[-1]), targets.view(-1))
        return logits, loss

    #claude generated
    @torch.no_grad()
    def generate(self, src, bos_id, max_new_tokens=64, temperature=0.8, top_k=40):
        self.eval()                                    # turns dropout off
        enc = self.encode(src)                         # (B, S, n_embd), computed once
        out = torch.full((src.shape[0], 1), bos_id, dtype=torch.long, device=src.device)

        for _ in range(max_new_tokens):
            dec_in = out[:, -block_size:]              # crop to the position table size
            logits = self.decode(dec_in, enc)[:, -1, :] / temperature     # last position only

            if top_k is not None:                      # keep only the k most likely tokens
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')

            probs = f.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            out = torch.cat([out, next_tok], dim=1)

        return out[:, 1:]                              # drop the start token

In [ ]:
m = BART().to(device)                         # your model class name
print(sum(p.numel() for p in m.parameters()) / 1e6, 'M parameters')

src, dec_in, tgt = get_batch('train')
logits, loss = m(src, dec_in, tgt)
print(logits.shape, loss.item())

4.290168 M parameters
torch.Size([32, 128, 3000]) 8.17811393737793


#GPT:


---
*   an autoregressive model with n_layers of **decoder** blocks  
*   requires a single input stream for decoder


In [ ]:
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embd = nn.Embedding(vocab_size, n_embd)
        self.pos_embd = nn.Embedding(block_size, n_embd)

        head_size = n_embd // n_head
        self.sa  = nn.ModuleList([MultiHead(head_size, n_head, causal=True) for _ in range(n_layers)])
        self.fwd = nn.ModuleList([FeedForward(n_embd) for _ in range(n_layers)])
        self.ln1 = nn.ModuleList([nn.LayerNorm(n_embd) for _ in range(n_layers)])
        self.ln2 = nn.ModuleList([nn.LayerNorm(n_embd) for _ in range(n_layers)])

        self.lnf = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embd(idx) + self.pos_embd(torch.arange(T, device=idx.device))

        for sa, fwd, ln1, ln2 in zip(self.sa, self.fwd, self.ln1, self.ln2):
            x = x + sa(ln1(x))
            x = x + fwd(ln2(x))

        logits = self.lm_head(self.lnf(x))          # (B, T, vocab_size)
        loss = None if targets is None else f.cross_entropy(
            logits.view(-1, logits.shape[-1]), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        self.eval()
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            probs = f.softmax(logits[:, -1, :] / temperature, dim=-1)
            idx = torch.cat((idx, torch.multinomial(probs, num_samples=1)), dim=1)
        return idx

In [ ]:
gpt = GPT().to(device)
print(sum(p.numel() for p in gpt.parameters()) / 1e6, 'M parameters')

src, dec_in, tgt = get_batch('train')
logits, loss = gpt(src, tgt)
print(logits.shape, loss.item())

3.845688 M parameters
torch.Size([32, 128, 3000]) 8.194321632385254


#BERT


---
*   a bidirectional model with n_layers of encoder block
*   requires a single stream masked input



In [ ]:
class BERT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embd = nn.Embedding(vocab_size, n_embd)
        self.pos_embd = nn.Embedding(block_size, n_embd)

        head_size = n_embd // n_head
        self.sa  = nn.ModuleList([MultiHead(head_size, n_head, causal=False) for _ in range(n_layers)])
        self.fwd = nn.ModuleList([FeedForward(n_embd) for _ in range(n_layers)])
        self.ln1 = nn.ModuleList([nn.LayerNorm(n_embd) for _ in range(n_layers)])
        self.ln2 = nn.ModuleList([nn.LayerNorm(n_embd) for _ in range(n_layers)])

        self.lnf = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embd(idx) + self.pos_embd(torch.arange(T, device=idx.device))

        for sa, fwd, ln1, ln2 in zip(self.sa, self.fwd, self.ln1, self.ln2):
            x = x + sa(ln1(x))
            x = x + fwd(ln2(x))

        logits = self.lm_head(self.lnf(x))          # (B, T, vocab_total)
        loss = None if targets is None else f.cross_entropy(
            logits.view(-1, logits.shape[-1]), targets.view(-1), ignore_index=-100)
        return logits, loss

In [ ]:
bert = BERT().to(device)
print(sum(p.numel() for p in bert.parameters()) / 1e6, 'M parameters')

src, dec_in, tgt = get_batch('train')
logits, loss = bert(src, tgt)
print(logits.shape, loss.item())

3.845688 M parameters
torch.Size([32, 128, 3000]) 8.196657180786133


#Output:


---



*   BART(with continuation)
*   GPT
*   BERT




In [ ]:
@torch.no_grad()
def estimate_loss():
    m.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            s, d_in, t = get_batch(split)
            _, loss = m(s, d_in, t)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    m.train()
    return out

optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)

for it in range(max_iters):
    if it % eval_interval == 0 or it == max_iters - 1:
        L = estimate_loss()
        print(f"step {it}: train {L['train']:.3f}, val {L['val']:.3f}")

    s, d_in, t = get_batch('train')
    _, loss = m(s, d_in, t)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train 8.185, val 8.183
step 500: train 4.221, val 4.256
step 1000: train 3.728, val 3.756
step 1500: train 3.475, val 3.510
step 2000: train 3.297, val 3.355
step 2500: train 3.178, val 3.232
step 3000: train 3.086, val 3.136
step 3500: train 3.014, val 3.073
step 4000: train 2.946, val 3.014
step 4500: train 2.892, val 2.960
step 5000: train 2.841, val 2.901
step 5500: train 2.795, val 2.854
step 6000: train 2.760, val 2.816
step 6500: train 2.716, val 2.798
step 7000: train 2.692, val 2.724
step 7500: train 2.664, val 2.720
step 8000: train 2.621, val 2.706
step 8500: train 2.603, val 2.692
step 9000: train 2.565, val 2.645
step 9500: train 2.548, val 2.621
step 10000: train 2.512, val 2.596
step 10500: train 2.497, val 2.561
step 11000: train 2.487, val 2.573
step 11500: train 2.472, val 2.547
step 12000: train 2.446, val 2.523
step 12500: train 2.428, val 2.514
step 13000: train 2.416, val 2.501
step 13500: train 2.405, val 2.476
step 14000: train 2.388, val 2.469
step 1450

In [ ]:
src, _, tgt = get_batch('val')
gen = m.generate(src[:1], bos_id, max_new_tokens=200)

print('SOURCE:\n', decode(src[0].tolist()))
print('\nACTUAL:\n', decode(tgt[0].tolist()))
print('\nMODEL:\n', decode(gen[0].tolist()))

SOURCE:
  and his muscle was hard. He stayed there for a long time until he felt better. Then he got up and went to play again. The bear learned that sometimes it's good to rest when your muscle hurts.Once upon a time, there was a little boy named Timmy. Timmy loved to play outside with his friends. One day, Timmy's mom said, "Timmy, mind the weather. It looks like it might rain soon."

Timmy looked up at the sky and saw some big, brown clouds. He knew that meant his mom was right. Timmy ran to his friends and said, "Hey gu

ACTUAL:
 ys, we better go inside. The weather is going to be bad soon."

His friends agreed and they all went inside to play. Just as they got inside, it started to rain really hard. Timmy's mom was happy that he listened to her and minded the weather.Once upon a time, there was a little boy named Timmy. Timmy loved to dance and twirl around. He would dance whenever he felt happy or excited. One day, Timmy saw an old man walking with a cane. The old man looked nerv

#GPT

In [ ]:
@torch.no_grad()
def estimate_loss_gpt(model):
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch_gpt(split)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out
optimizer = torch.optim.AdamW(gpt.parameters(), lr=learning_rate)

for it in range(max_iters):
    x, y = get_batch_gpt('train')
    _, loss = gpt(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(gpt.parameters(), 1.0)
    optimizer.step()

    if it % eval_interval == 0 or it == max_iters - 1:
        L = estimate_loss_gpt(gpt)
        print(f"step {it}: train {L['train']:.3f}, val {L['val']:.3f}")

step 0: train 7.967, val 7.972
step 500: train 4.043, val 4.081
step 1000: train 3.577, val 3.619
step 1500: train 3.317, val 3.378
step 2000: train 3.175, val 3.195
step 2500: train 3.047, val 3.097
step 3000: train 2.972, val 3.015
step 3500: train 2.898, val 2.950
step 4000: train 2.825, val 2.882
step 4500: train 2.770, val 2.828
step 5000: train 2.725, val 2.784
step 5500: train 2.674, val 2.734
step 6000: train 2.647, val 2.709
step 6500: train 2.606, val 2.678
step 7000: train 2.573, val 2.647
step 7500: train 2.554, val 2.623
step 8000: train 2.524, val 2.583
step 8500: train 2.497, val 2.581
step 9000: train 2.471, val 2.546
step 9500: train 2.454, val 2.530
step 10000: train 2.443, val 2.525
step 10500: train 2.417, val 2.496
step 11000: train 2.404, val 2.466
step 11500: train 2.382, val 2.461
step 12000: train 2.378, val 2.453
step 12500: train 2.354, val 2.421
step 13000: train 2.356, val 2.419
step 13500: train 2.333, val 2.400
step 14000: train 2.309, val 2.397
step 1450

In [ ]:
src, _, tgt = get_batch('val')
gen = m.generate(src[:1], bos_id, max_new_tokens=200)

print('SOURCE:\n', decode(src[0].tolist()))
print('\nACTUAL:\n', decode(tgt[0].tolist()))
print('\nMODEL:\n', decode(gen[0].tolist()))

SOURCE:
  because it had a big bookshelf. The bookshelf was full of colorful books. Mia liked to read these books every day.

One day, Mia was very tired. She wanted to find a book to read before she went to bed. Mia looked at the bookshelf and picked out a book with a pretty cover. She sat down and started to read the book.

As Mia read the book, she began to feel more and more sleepy. She read and read until she reached the end of the story. Mia was so tired that she closed the book and put it back on the bookshelf.

ACTUAL:
  Then, she went to bed and had sweet dreams.Once upon a time, there was a little bunny named Fluffy. Fluffy loved to play in the garden and eat carrots. One day, while Fluffy was playing, he saw a big hole in the fence. Fluffy wanted to escape and explore the world outside the garden.

Fluffy went to his friend, a little bird named Tweety. "Tweety, I want to escape and see the world outside the garden. Do you want to come with me?" asked Fluffy. "Yes, I want to 

#BERT

In [ ]:
@torch.no_grad()
def estimate_bert(model):
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses, accs = torch.zeros(eval_iters), torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, _ = get_batch_gpt(split)
            inp, tgt = mask_batch(x)
            logits, loss = model(inp, tgt)
            sel = tgt != -100
            accs[k] = (logits.argmax(-1)[sel] == tgt[sel]).float().mean()
            losses[k] = loss.item()
        out[split] = (losses.mean().item(), accs.mean().item())
    model.train()
    return out



optimizer = torch.optim.AdamW(bert.parameters(), lr=learning_rate)   # same cell as the model

for it in range(max_iters):
    if it % eval_interval == 0 or it == max_iters - 1:
        L = estimate_bert(bert)
        print(f"step {it}: train {L['train'][0]:.3f} (acc {L['train'][1]:.3f}), "
              f"val {L['val'][0]:.3f} (acc {L['val'][1]:.3f})")

    x, _ = get_batch_gpt('train')
    inp, tgt = mask_batch(x)
    _, loss = bert(inp, tgt)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(bert.parameters(), 1.0)
    optimizer.step()

step 0: train 8.224 (acc 0.000), val 8.224 (acc 0.000)
step 500: train 5.992 (acc 0.068), val 6.029 (acc 0.064)
step 1000: train 5.904 (acc 0.070), val 5.959 (acc 0.065)
step 1500: train 5.877 (acc 0.069), val 5.910 (acc 0.066)
step 2000: train 5.848 (acc 0.070), val 5.919 (acc 0.063)
step 2500: train 5.848 (acc 0.069), val 5.884 (acc 0.064)
step 3000: train 5.820 (acc 0.069), val 5.848 (acc 0.064)
step 3500: train 5.803 (acc 0.073), val 5.835 (acc 0.068)
step 4000: train 5.567 (acc 0.097), val 5.620 (acc 0.090)
step 4500: train 4.963 (acc 0.166), val 5.017 (acc 0.163)
step 5000: train 4.599 (acc 0.201), val 4.664 (acc 0.196)
step 5500: train 4.332 (acc 0.235), val 4.369 (acc 0.233)
step 6000: train 3.995 (acc 0.279), val 4.032 (acc 0.276)
step 6500: train 3.741 (acc 0.311), val 3.803 (acc 0.307)
step 7000: train 3.559 (acc 0.336), val 3.610 (acc 0.331)
step 7500: train 3.401 (acc 0.354), val 3.468 (acc 0.349)
step 8000: train 3.305 (acc 0.368), val 3.327 (acc 0.363)
step 8500: train 3